#### AI Dataset Insight Generator - Full Experimentation Notebook

This notebook:
- Loads a CSV dataset
- Inspects its structure
- Detects numeric-like columns safely
- Cleans numeric columns
- Generates numeric summary
- Generates categorical/text summary
- Produces AI-ready JSON summary

Features:
- Safe numeric detection avoids misclassifying IDs or mostly-text columns
- Provides descriptive analysis for non-numeric columns
- Modular, clean functions ready for production

## LIBRARY IMPORTS

In [19]:
import pandas as pd
import numpy as np
import json

# Display options
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

## Global Variables

In [21]:
# Dataset path
DATASET_PATH = "amazon.csv"

# Preview rows
PREVIEW_ROWS = 5

# Threshold for numeric detection (90% of rows must be numeric)
NUMERIC_THRESHOLD = 0.9

## Helper Functions

In [24]:
def load_dataset(file_path: str) -> pd.DataFrame:
    """Load dataset from CSV safely."""
    try:
        df = pd.read_csv(file_path)
        print("✅ Dataset loaded successfully")
        return df
    except FileNotFoundError:
        raise Exception(f"❌ File not found: {file_path}")
    except Exception as error:
        raise Exception(f"❌ Error loading dataset: {error}")


def inspect_dataset(df: pd.DataFrame) -> dict:
    """Inspect dataset structure and missing values."""
    return {
        "rows": df.shape[0],
        "columns": df.shape[1],
        "column_names": list(df.columns),
        "data_types": df.dtypes.astype(str).to_dict(),
        "missing_values": df.isnull().sum().to_dict()
    }


def detect_numeric_columns_safe(df: pd.DataFrame, threshold: float = 0.8) -> list:
    """
    Detect numeric-like columns safely, ignoring mostly-text columns.

    threshold: proportion of values that must be numeric
    """
    numeric_cols = []
    for col in df.columns:
        col_series = df[col].astype(str)
        cleaned = col_series.str.replace('[^\d\.\-]', '', regex=True)
        is_numeric = pd.to_numeric(cleaned, errors='coerce').notnull()
        ratio = is_numeric.sum() / len(col_series)
        if ratio >= threshold:
            numeric_cols.append(col)
    return numeric_cols


def clean_numeric_columns_safe(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """Convert numeric-like columns to float safely."""
    for col in columns:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace('[^\d\.\-]', '', regex=True),
            errors='coerce'
        )
    return df


def numeric_summary(df: pd.DataFrame) -> dict:
    """Generate descriptive statistics for numeric columns."""
    numeric_cols = df.select_dtypes(include=["number"])
    if numeric_cols.empty:
        return {}
    return numeric_cols.describe().to_dict()


def categorical_summary(df: pd.DataFrame) -> dict:
    """
    Generate summary for non-numeric columns:
    - Number of unique values
    - Top 5 most frequent values
    """
    cat_cols = df.select_dtypes(exclude=["number"])
    summary = {}
    for col in cat_cols:
        top_values = df[col].value_counts().head(5).to_dict()
        summary[col] = {
            "unique_count": df[col].nunique(),
            "top_values": top_values
        }
    return summary


def generate_dataset_summary_full(df: pd.DataFrame, exclude_cols: list = []) -> dict:
    """Generate full dataset summary including numeric and categorical analysis."""
    # Detect numeric-like columns
    numeric_cols = detect_numeric_columns_safe(df, threshold=NUMERIC_THRESHOLD)
    # Exclude known ID/text columns
    numeric_cols = [c for c in numeric_cols if c not in exclude_cols]
    # Clean numeric columns
    df = clean_numeric_columns_safe(df, numeric_cols)
    
    return {
        "metadata": inspect_dataset(df),
        "numeric_summary": numeric_summary(df),
        "categorical_summary": categorical_summary(df)
    }


def summary_to_json(summary: dict) -> str:
    """Convert dataset summary to formatted JSON."""
    return json.dumps(summary, indent=2)

## Load Dataset

In [25]:
df = load_dataset(DATASET_PATH)
print(f"Dataset shape: {df.shape}")
df.head(PREVIEW_ROWS)

✅ Dataset loaded successfully
Dataset shape: (1465, 16)


,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,₹399,"₹1,099",64%,4.2,"24,269",High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,₹199,₹349,43%,4.0,"43,994","Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,₹199,"₹1,899",90%,3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,₹329,₹699,53%,4.2,"94,363",The boAt Deuce USB 300 2 in 1 cable is compati...,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...","Omkar dhale,JD,HEMALATHA,Ajwadh a.,amar singh ...","R3EEUZKKK9J36I,R3HJVYCLYOY554,REDECAZ7AMPQC,R1...","Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou...",https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,₹154,₹399,61%,4.2,"16,905",[CHARGE & SYNC FUNCTION]- This cable comes wit...,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...","rahuls6099,Swasat Borah,Ajay Wadke,Pranali,RVK...","R1BP4L2HH9TFUP,R16PVJEXKV6QZS,R2UPDB81N66T4P,R...","As good as original,Decent,Good one for second...","Bought this instead of original apple, does th...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...


## Inspect Dataset

In [26]:
dataset_info = inspect_dataset(df)
dataset_info

{'rows': 1465,
 'columns': 16,
 'column_names': ['product_id',
  'product_name',
  'category',
  'discounted_price',
  'actual_price',
  'discount_percentage',
  'rating',
  'rating_count',
  'about_product',
  'user_id',
  'user_name',
  'review_id',
  'review_title',
  'review_content',
  'img_link',
  'product_link'],
 'data_types': {'product_id': 'object',
  'product_name': 'object',
  'category': 'object',
  'discounted_price': 'object',
  'actual_price': 'object',
  'discount_percentage': 'object',
  'rating': 'object',
  'rating_count': 'object',
  'about_product': 'object',
  'user_id': 'object',
  'user_name': 'object',
  'review_id': 'object',
  'review_title': 'object',
  'review_content': 'object',
  'img_link': 'object',
  'product_link': 'object'},
 'missing_values': {'product_id': 0,
  'product_name': 0,
  'category': 0,
  'discounted_price': 0,
  'actual_price': 0,
  'discount_percentage': 0,
  'rating': 0,
  'rating_count': 2,
  'about_product': 0,
  'user_id': 0,
  'u

## Detect & Clean Numeric Columns Safely

In [29]:
#  Detect numeric-like columns safely
numeric_cols = detect_numeric_columns_safe(df, threshold=NUMERIC_THRESHOLD)
print(f"Detected numeric columns: {numeric_cols}")

# Dynamically exclude columns containing "_id"
numeric_cols = [c for c in numeric_cols if "_id" not in c.lower()]
print(f"Numeric columns after excluding '_id': {numeric_cols}")

# Clean numeric columns
df = clean_numeric_columns_safe(df, numeric_cols)

# Validate cleaning
df[numeric_cols].head(PREVIEW_ROWS)
df.dtypes

Detected numeric columns: ['product_id', 'discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count', 'user_id', 'review_id']
Numeric columns after excluding '_id': ['discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count']


product_id              object
product_name            object
category                object
discounted_price       float64
actual_price           float64
discount_percentage      int64
rating                 float64
rating_count           float64
about_product           object
user_id                 object
user_name               object
review_id               object
review_title            object
review_content          object
img_link                object
product_link            object
dtype: object

## GENERATE FULL SUMMARY (NUMERIC AND CATEGORICAL)

In [31]:
# Columns to exclude from numeric detection (IDs, names, categories)
exclude_cols = ['product_id','review_id','user_id','user_name','product_name','category']

# Generate full dataset summary
full_summary = generate_dataset_summary_full(df, exclude_cols=exclude_cols)

# Preview summary
full_summary

{'metadata': {'rows': 1465,
  'columns': 16,
  'column_names': ['product_id',
   'product_name',
   'category',
   'discounted_price',
   'actual_price',
   'discount_percentage',
   'rating',
   'rating_count',
   'about_product',
   'user_id',
   'user_name',
   'review_id',
   'review_title',
   'review_content',
   'img_link',
   'product_link'],
  'data_types': {'product_id': 'object',
   'product_name': 'object',
   'category': 'object',
   'discounted_price': 'float64',
   'actual_price': 'float64',
   'discount_percentage': 'int64',
   'rating': 'float64',
   'rating_count': 'float64',
   'about_product': 'object',
   'user_id': 'object',
   'user_name': 'object',
   'review_id': 'object',
   'review_title': 'object',
   'review_content': 'object',
   'img_link': 'object',
   'product_link': 'object'},
  'missing_values': {'product_id': 0,
   'product_name': 0,
   'category': 0,
   'discounted_price': 0,
   'actual_price': 0,
   'discount_percentage': 0,
   'rating': 1,
   'rat

## CONVERT SUMMARY TO JSON

In [32]:
json_summary = summary_to_json(full_summary)
print(json_summary)

{
  "metadata": {
    "rows": 1465,
    "columns": 16,
    "column_names": [
      "product_id",
      "product_name",
      "category",
      "discounted_price",
      "actual_price",
      "discount_percentage",
      "rating",
      "rating_count",
      "about_product",
      "user_id",
      "user_name",
      "review_id",
      "review_title",
      "review_content",
      "img_link",
      "product_link"
    ],
    "data_types": {
      "product_id": "object",
      "product_name": "object",
      "category": "object",
      "discounted_price": "float64",
      "actual_price": "float64",
      "discount_percentage": "int64",
      "rating": "float64",
      "rating_count": "float64",
      "about_product": "object",
      "user_id": "object",
      "user_name": "object",
      "review_id": "object",
      "review_title": "object",
      "review_content": "object",
      "img_link": "object",
      "product_link": "object"
    },
    "missing_values": {
      "product_id": 0,
   